# DigitalBookLLM: Adaptive RAG Pipeline

This notebook implements an **adaptive, dual-mode RAG system**. It intelligently handles two common user scenarios:

1.  **Prioritized RAG (Tutor Mode):** When a user provides both a `query` AND `selected_text`. The system provides a synthesized, conversational answer anchored to the user's selection.
2.  **Standard RAG (Search Mode):** When a user provides *only* a `query`. The system searches the entire document for a general answer.

This approach combines the strengths of a context-aware tutor with a general document search, all in one pipeline.

**Pipeline:**
1.  **Setup:** Install libraries and set API keys.
2.  **Load Data:** Download a source file (`.txt` book).
3.  **Chunk & Index:** Split the document and embed it in a FAISS vector store.
4.  **Adaptive RAG:** Define the core logic that routes between "Tutor Mode" and "Search Mode".
5.  **Query:** Run examples for both modes.

In [ ]:
# 1. Install necessary libraries
# %pip install -q gdown together langchain-text-splitters sentence-transformers faiss-cpu python-dotenv

In [ ]:
# 2. Import libraries
import os
import getpass
import gdown
from together import Together
import faiss
import numpy as np
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

## Section 1: Setup and Data Loading

First, we'll set up the Together AI API key. Then, we will use `gdown` to download our source document (Alice's Adventures in Wonderland).

In [ ]:
# 3. Set up Together AI API Key

# Try to load from .env file first (if you have one)
load_dotenv()

if "TOGETHER_API_KEY" not in os.environ:
    os.environ["TOGETHER_API_KEY"] = getpass.getpass("Enter your Together AI API Key: ")

# Create Together client
together_client = Together(api_key=os.environ["TOGETHER_API_KEY"])

print("Together AI API Key set.")

In [ ]:
# 4. Download Source File using gdown
file_url = "https://www.gutenberg.org/files/11/11-0.txt"
output_filename = "source_document.txt"

if not os.path.exists(output_filename):
    print(f"Downloading {output_filename}...")
    gdown.download(file_url, output_filename, quiet=False)
else:
    print(f"'{output_filename}' already exists.")

# 5. Load the document into memory
try:
    with open(output_filename, 'r', encoding='utf-8') as f:
        full_document = f.read()
    print(f"\nSuccessfully loaded document ({len(full_document)} characters).")
    
    # Show a snippet (skipping the Gutenberg header)
    print("\n--- Document Snippet ---")
    print(full_document[3000:3500])
except FileNotFoundError:
    print(f"Error: File '{output_filename}' not found. Download may have failed.")
except Exception as e:
    print(f"An error occurred: {e}")

## Section 2: Chunking, Embedding, and Indexing

This section is standard RAG setup. We process the loaded text, split it into chunks, create vector embeddings for each chunk, and store them in a FAISS in-memory index for fast retrieval.

In [ ]:
# 6. Chunk the document
print("Chunking document...")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # Aim for chunks of this size
    chunk_overlap=150, # Overlap helps maintain context
    length_function=len,
)

doc_chunks = text_splitter.split_text(full_document)

print(f"Document split into {len(doc_chunks)} chunks.")
print(f"--- Example Chunk 0 ---\n{doc_chunks[0]}")

In [ ]:
# 7. Initialize Embedding Model (Local)
print("Loading embedding model (all-MiniLM-L6-v2). This may take a moment...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model loaded.")

embedding_dim = embedding_model.get_sentence_embedding_dimension()
print(f"Embedding dimension: {embedding_dim}")

In [ ]:
# 8. Create FAISS Index
index = faiss.IndexFlatL2(embedding_dim)
print(f"Created FAISS index with dimension {embedding_dim}.")

chunk_id_to_text = {i: text for i, text in enumerate(doc_chunks)}

print("Embedding all chunks...")
chunk_embeddings = embedding_model.encode(doc_chunks)

index.add(chunk_embeddings.astype('float32')) # FAISS requires float32

print(f"Added {index.ntotal} vectors to the index. System is ready.")

## Section 3: The Adaptive RAG Pipeline (Core Logic)

This is the core of our improved system. The single function `run_adaptive_rag` now checks *what* the user provided.

* If **both `query` and `selected_text`** are given, it runs the "Prioritized RAG" (Tutor Mode) flow using the conversational, synthesizing prompt.
* If **only a `query`** is given, it runs a "Standard RAG" (Search Mode) flow to answer the general question.

This 'agentic' decision step makes the chat interface much more flexible.

In [ ]:
# 9. Define the Adaptive RAG function (v2)

def run_adaptive_rag(query: str, selected_text: str = None, top_k: int = 3, llm_model: str = "meta-llama/Llama-3.3-70B-Instruct-Turbo-Free"):
    """
    Runs an adaptive RAG pipeline.
    
    - If selected_text is provided, it runs in "Tutor Mode" (Prioritized RAG).
    - If selected_text is None, it runs in "Search Mode" (Standard RAG).
    """
    
    print("--- New Query Received ---")
    print(f"Query: {query}")
    
    prompt_template = ""
    retrieved_chunks = []

    if selected_text:
        # --- MODE 1: Prioritized RAG (Tutor Mode) --- 
        print("Mode: Prioritized RAG (Tutor Mode)")
        print(f"Selected Text (Anchor):\n'{selected_text}'")
        print("-" * 28)
        
        # 1.a. Augmented Retrieval (Prioritized)
        search_query = query + " " + selected_text
        query_embedding = embedding_model.encode([search_query])
        distances, indices = index.search(query_embedding.astype('float32'), top_k)
        
        print(f"Retrieving {top_k} related chunks...")
        for i, idx in enumerate(indices[0]):
            retrieved_text = chunk_id_to_text.get(idx)
            
            # 1.b. Context De-duplication
            if retrieved_text.strip() != selected_text.strip():
                retrieved_chunks.append(retrieved_text)
                print(f"  -> Related Chunk {i+1} (Dist: {distances[0][i]:.4f}): {retrieved_text[:100]}...")
            else:
                print(f"  -> (Skipped duplicate chunk: {idx})")

        related_context_str = "\n---\n".join(retrieved_chunks) if retrieved_chunks else "None"

        # 1.c. Prompt Construction (Tutor-Style)
        prompt_template = f"""You are an AI study buddy. Your goal is to provide a clear, compact, and helpful answer to the user's question.

You have two sources of information:
1.  **[USER'S SELECTION]**: The specific text the user highlighted. This is your main anchor point.
2.  **[RELATED CONTEXT]**: Additional snippets from the document for background.

**Your Task:**
* Directly answer the [USER'S QUESTION] in a natural, conversational tone.
* Ground your answer in the [USER'S SELECTION].
* Naturally *synthesize* information from the [RELATED CONTEXT] only as needed to provide a more complete picture.
* **Do not** use robotic phrases like "Based on the primary context...". Just weave the information together.
* If the contexts don't contain the answer, state that.

---[USER'S SELECTION]---
{selected_text}
---[END USER'S SELECTION]---

---[RELATED CONTEXT]---
{related_context_str}
---[END RELATED CONTEXT]---

[USER QUESTION]
{query}

**Helpful Answer:**
"""

    else:
        # --- MODE 2: Standard RAG (Search Mode) --- 
        print("Mode: Standard RAG (Search Mode)")
        print("-" * 28)

        # 2.a. Standard Retrieval
        search_query = query
        query_embedding = embedding_model.encode([search_query])
        distances, indices = index.search(query_embedding.astype('float32'), top_k)
        
        print(f"Retrieving {top_k} relevant chunks...")
        for i, idx in enumerate(indices[0]):
            retrieved_chunks.append(chunk_id_to_text.get(idx))
            print(f"  -> Relevant Chunk {i+1} (Dist: {distances[0][i]:.4f}): {retrieved_chunks[-1][:100]}...")

        context_str = "\n---\n".join(retrieved_chunks) if retrieved_chunks else "None"

        # 2.b. Prompt Construction (Standard)
        prompt_template = f"""You are a helpful AI assistant. Answer the user's question based *only* on the provided context.
        
Keep your answer clear and concise. If the answer is not in the context, say so.

---[CONTEXT]---
{context_str}
---[END CONTEXT]---

[USER QUESTION]
{query}

**Answer:**
"""

    # --- 4. Call Together AI LLM (Common Step) ---
    print(f"\nCalling Together AI ({llm_model})...")
    
    try:
        response = together_client.chat.completions.create(
            model=llm_model,
            messages=[
                {"role": "user", "content": prompt_template} 
            ],
            temperature=0.2,  # Low, but allows for more natural language
            max_tokens=750,
        )
        
        final_answer = response.choices[0].message.content
        return final_answer
    except Exception as e:
        print(f"Error calling Together AI: {e}")
        return "Error: Could not generate an answer."

## Section 4: Running Examples

Let's test both modes of our adaptive system.

* **Examples 1-3:** Test "Tutor Mode" (query + selection)
* **Example 4:** Tests "Search Mode" (query only)

In [ ]:
# --- Example 1: Tutor Mode (Answer in selection) ---

sel_text_1 = "The rabbit-hole went straight on like a tunnel for some way, and then dipped suddenly down, so suddenly that Alice had not a moment to think about stopping herself before she found herself falling down what seemed to be a very deep well."
query_1 = "What happened to Alice at the rabbit-hole, and what did it seem like?"

answer_1 = run_adaptive_rag(query_1, sel_text_1)
print("\n=== FINAL ANSWER 1 ===\n", answer_1)

In [ ]:
# --- Example 2: Tutor Mode (Answer requires synthesis) ---

sel_text_2 = "Soon her eye fell on a little glass box that was lying under the table: she opened it, and found in it a very small cake, on which the words 'EAT ME' were beautifully marked in currants."
query_2 = "What was the plan for the 'EAT ME' cake?"

# The selection only says *what* it was, not the *plan*.
# The related context (retrieved chunks) should contain the plan.
answer_2 = run_adaptive_rag(query_2, sel_text_2, top_k=2)
print("\n=== FINAL ANSWER 2 ===\n", answer_2)

In [ ]:
# --- Example 3: Tutor Mode (Distant query) ---

# The user selects text about the Rabbit.
sel_text_3 = "Alice started to her feet, for it flashed across her mind that she had never before seen a rabbit with either a waistcoat-pocket, or a watch to take out of it..."

# The question is about something else entirely.
query_3 = "What did Alice find on the table in the great hall?"

# The RAG system should find the chunk about the "golden key".
# The LLM should follow instructions and answer using the *Related Context*,
# as the User's Selection isn't relevant to the question.

answer_3 = run_adaptive_rag(query_3, sel_text_3, top_k=2)
print("\n=== FINAL ANSWER 3 ===\n", answer_3)

In [ ]:
# --- Example 4: Search Mode (Query Only) ---

# The user highlights NO text and asks a general question.
query_4 = "What was the Mock Turtle's story about?"

# The system should automatically enter 'Search Mode'.
answer_4 = run_adaptive_rag(query_4, selected_text=None, top_k=3)
print("\n=== FINAL ANSWER 4 ===\n", answer_4)

## Section 5: Scoring and Metrics (Updated)

To evaluate this *adaptive* system, you need to test both modes.

### 1. Retrieval Metrics (for both modes)

* **Context Precision @ K:** Of the `K` chunks retrieved, how many were *actually relevant* to answering the `query`? (This is the most important retrieval metric).

### 2. Generation Metrics (The RAG Triad, adapted)

1.  **Faithfulness (Groundedness):**
    * **Question:** "Given the *provided context(s)*, is the generated answer *fully supported* by them? Is there any hallucinated or contradictory information?"
    * **Applies to:** Both modes.

2.  **Answer Relevance:**
    * **Question:** "Given the *user's question*, does the generated answer *directly and completely* address it?"
    * **Applies to:** Both modes.

3.  **Synthesis Quality (for Tutor Mode only):**
    * **Question:** "Does the answer *naturally weave together* the 'User's Selection' and 'Related Context'? Does it avoid robotic phrasing and feel like a single, cohesive explanation?"
    * **Applies to:** Only Tutor Mode (when `selected_text` is provided).